# Scaled English-to-Odia Transformer (Enhanced Architecture)
## Accelerated Kaggle GPU Training (T4 GPU + FP16 AMP)

This notebook trains a **Scaled Transformer** (~14.5M parameters with weight tying) in **~10–12 minutes** on Kaggle.

### IMPORTANT SETTINGS in Kaggle Sidebar (Notebook Options):
1. **Accelerator**: Select **`GPU T4 x 2`** (Do NOT choose P100; P100 is incompatible with PyTorch and crashes).
2. **Internet**: Toggle to **`On`** (required to stream `ai4bharat/samanantar`).

---

### Step 1: Verify Hardware Accelerator (T4 GPU Required)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ERROR: GPU is not enabled! In Kaggle sidebar -> Notebook options -> Accelerator -> Select 'GPU T4 x 2'.")

cap = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)

if cap[0] < 7:
    raise RuntimeError(
        f"CRITICAL ERROR: Assigned GPU is '{gpu_name}' (Capability {cap[0]}.{cap[1]} < 7.0)!\n"
        "This older card causes PyTorch kernel failures on Kaggle.\n"
        "FIX: In the right sidebar -> Notebook options -> Accelerator -> Select 'GPU T4 x 2'!"
    )

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"SUCCESS: Connected to {gpu_name} (Capability {cap[0]}.{cap[1]}) with {vram_gb:.1f} GB VRAM!")
print("Turing Tensor Cores and FP16 Mixed Precision are fully operational.")

### Step 2: Install Dependencies

In [ ]:
!pip install -q datasets tokenizers sacrebleu

### Step 3: Enhanced Model Hyperparameters

In [ ]:
import os
import math
import time
import json
import copy
import unicodedata
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Output directory on Kaggle
KAGGLE_WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

# Architecture (Option A: 14.5M Params with weight tying)
D_MODEL = 256
N_HEADS = 8
D_FF = 1024
N_ENCODER_LAYERS = 4
N_DECODER_LAYERS = 4
DROPOUT = 0.1
MAX_LEN = 96  # Subwords including <SOS>/<EOS>
TIE_WEIGHTS = True

# Vocabularies
EN_VOCAB_SIZE = 8000
OR_VOCAB_SIZE = 8000
SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

# Dataset targets
CANDIDATE_POOL = 75000
TRAIN_SIZE = 60000
VAL_SIZE = 2000
TEST_SIZE = 2000
TOTAL_SIZE = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

# Training & Optimization
MICRO_BATCH_SIZE = 128
ACCUM_STEPS = 2  # Effective batch size = 256
NUM_EPOCHS = 25
BASE_LR = 5e-4
MIN_LR = 1e-6
WARMUP_STEPS = 1200
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
LABEL_SMOOTHING = 0.1

DEVICE = torch.device("cuda")
print("Hyperparameters configured.")

### Step 4: Text Cleaning & Data Pipeline
Streams English–Odia pairs from `ai4bharat/samanantar` using `trust_remote_code=True`, applies Unicode NFC normalization, removes zero-width artifacts, and preserves Indic conjuncts (ZWJ/ZWNJ).

In [ ]:
from datasets import load_dataset

_ZWSP = "\u200B"
_BOM = "\uFEFF"
_ZWJ = "\u200D"
_ZWNJ = "\u200C"
_ZW_JOINERS = _ZWJ + _ZWNJ
_EDGE_JOINER_RUN = re.compile(f"^[{_ZW_JOINERS}]+|[{_ZW_JOINERS}]+$")
_INTERIOR_JOINER_RUN = re.compile(f"[{_ZW_JOINERS}]{{2,}}")
_WHITESPACE_RUN = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not text: return ""
    text = unicodedata.normalize("NFC", text)
    text = text.replace(_ZWSP, "").replace(_BOM, "")
    text = _EDGE_JOINER_RUN.sub("", text)
    text = _INTERIOR_JOINER_RUN.sub(lambda m: m.group(0)[0], text)
    return _WHITESPACE_RUN.sub(" ", text).strip()

print("Connecting to ai4bharat/samanantar (config: 'or')...")
try:
    dataset_stream = load_dataset("ai4bharat/samanantar", "or", split="train", streaming=True)
except Exception as e:
    print("\nFAILED TO LOAD DATASET: Please ensure 'Internet' is toggled to 'On' in Kaggle's right sidebar -> Notebook options -> Internet!")
    raise e

candidates = []
seen_src = set()
for row in dataset_stream:
    src = clean_text(row["src"])
    tgt = clean_text(row["tgt"])
    if not src or not tgt or src in seen_src: continue
    n_src, n_tgt = len(src.split()), len(tgt.split())
    if not (3 <= n_src <= 60 and 3 <= n_tgt <= 60): continue
    seen_src.add(src)
    candidates.append({"src": src, "tgt": tgt})
    if len(candidates) >= CANDIDATE_POOL:
        break

print(f"Collected {len(candidates):,} candidate pairs!")

### Step 5: Train Subword Tokenizers (Byte-Level BPE)
Trains 8,000-vocabulary BPE tokenizers with automatic `<SOS>` and `<EOS>` wrapping.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors, trainers

def train_bpe(texts, vocab_size):
    tok = Tokenizer(models.BPE(unk_token="<UNK>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tok.train_from_iterator(texts, trainer=trainer)
    tok.post_processor = processors.TemplateProcessing(
        single="<SOS> $A <EOS>",
        special_tokens=[("<SOS>", tok.token_to_id("<SOS>")), ("<EOS>", tok.token_to_id("<EOS>"))],
    )
    return tok

en_tok = train_bpe([c["src"] for c in candidates], EN_VOCAB_SIZE)
or_tok = train_bpe([c["tgt"] for c in candidates], OR_VOCAB_SIZE)

# Filter by MAX_LEN=96
filtered_pairs = []
for c in candidates:
    en_ids = en_tok.encode(c["src"]).ids
    or_ids = or_tok.encode(c["tgt"]).ids
    if len(en_ids) <= MAX_LEN and len(or_ids) <= MAX_LEN:
        filtered_pairs.append({"src": c["src"], "tgt": c["tgt"], "src_ids": en_ids, "tgt_ids": or_ids})

print(f"Survivors at MAX_LEN={MAX_LEN}: {len(filtered_pairs):,} ({len(filtered_pairs)/len(candidates):.1%} retention)")

# Create splits
np.random.seed(42)
np.random.shuffle(filtered_pairs)
final_pairs = filtered_pairs[:TOTAL_SIZE]

train_data = final_pairs[:TRAIN_SIZE]
val_data = final_pairs[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
test_data = final_pairs[TRAIN_SIZE + VAL_SIZE : TOTAL_SIZE]
print(f"Splits: Train={len(train_data):,}, Val={len(val_data):,}, Test={len(test_data):,}")

### Step 6: Pre-LayerNorm Transformer with Weight Tying

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class PositionalEncoding(nn.Module):
    def __init__(self, d_model=D_MODEL, dropout=DROPOUT, max_len=MAX_LEN + 16):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(1), :].unsqueeze(0))

class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout)
        self.scale = math.sqrt(d_model)

    def forward(self, ids):
        return self.pos_encoding(self.token_embedding(ids) * self.scale)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, Lq, _ = q.shape
        _, Lk, _ = k.shape
        q = self.q_proj(q).view(B, Lq, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.k_proj(k).view(B, Lk, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.v_proj(v).view(B, Lk, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e4)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).permute(0, 2, 1, 3).contiguous().view(B, Lq, self.d_model)
        return self.out_proj(out)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=D_MODEL, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

class PreLNEncoderBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        norm_x = self.norm1(x)
        x = x + self.dropout1(self.self_attn(norm_x, norm_x, norm_x, src_mask))
        norm_x = self.norm2(x)
        x = x + self.dropout2(self.ffn(norm_x))
        return x

class PreLNDecoderBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.dropout2 = nn.Dropout(dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask, src_mask):
        norm_x = self.norm1(x)
        x = x + self.dropout1(self.self_attn(norm_x, norm_x, norm_x, tgt_mask))
        norm_x = self.norm2(x)
        x = x + self.dropout2(self.cross_attn(norm_x, enc_out, enc_out, src_mask))
        norm_x = self.norm3(x)
        x = x + self.dropout3(self.ffn(norm_x))
        return x

class EnhancedScaledTransformer(nn.Module):
    def __init__(self, tie_weights=TIE_WEIGHTS):
        super().__init__()
        self.encoder_emb = Embeddings(EN_VOCAB_SIZE, D_MODEL, DROPOUT)
        self.decoder_emb = Embeddings(OR_VOCAB_SIZE, D_MODEL, DROPOUT)
        self.encoder_blocks = nn.ModuleList([PreLNEncoderBlock() for _ in range(N_ENCODER_LAYERS)])
        self.encoder_final_norm = nn.LayerNorm(D_MODEL)
        self.decoder_blocks = nn.ModuleList([PreLNDecoderBlock() for _ in range(N_DECODER_LAYERS)])
        self.decoder_final_norm = nn.LayerNorm(D_MODEL)
        self.output_proj = nn.Linear(D_MODEL, OR_VOCAB_SIZE, bias=False)

        if tie_weights:
            self.output_proj.weight = self.decoder_emb.token_embedding.weight

    def make_src_mask(self, src_ids):
        return (src_ids != PAD_ID).unsqueeze(1).unsqueeze(1)

    def make_tgt_mask(self, tgt_ids):
        T = tgt_ids.size(1)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt_ids.device))
        pad = (tgt_ids != PAD_ID).unsqueeze(1).unsqueeze(1)
        return causal.unsqueeze(0).unsqueeze(0) & pad

    def forward(self, src_ids, tgt_ids):
        src_mask = self.make_src_mask(src_ids)
        tgt_mask = self.make_tgt_mask(tgt_ids)
        x = self.encoder_emb(src_ids)
        for b in self.encoder_blocks:
            x = b(x, src_mask)
        x = self.encoder_final_norm(x)

        y = self.decoder_emb(tgt_ids)
        for b in self.decoder_blocks:
            y = b(y, x, tgt_mask, src_mask)
        y = self.decoder_final_norm(y)
        return self.output_proj(y)

model = EnhancedScaledTransformer().to(DEVICE)
total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"=== Model Instantiated: {total_p:,} Trainable Parameters (Pre-LN + Weight Tying) ===")

### Step 7: DataLoader & Cosine Warmup Schedule

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        return torch.tensor(self.pairs[idx]["src_ids"], dtype=torch.long), torch.tensor(self.pairs[idx]["tgt_ids"], dtype=torch.long)

def pad_collate(batch):
    src_list, tgt_list = zip(*batch)
    max_src = max(s.size(0) for s in src_list)
    max_tgt = max(t.size(0) for t in tgt_list)
    src_padded = torch.full((len(src_list), max_src), PAD_ID, dtype=torch.long)
    tgt_padded = torch.full((len(tgt_list), max_tgt), PAD_ID, dtype=torch.long)
    for i, s in enumerate(src_list): src_padded[i, :s.size(0)] = s
    for i, t in enumerate(tgt_list): tgt_padded[i, :t.size(0)] = t
    return src_padded, tgt_padded

train_loader = DataLoader(TranslationDataset(train_data), batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=pad_collate, num_workers=2, pin_memory=True)
val_loader = DataLoader(TranslationDataset(val_data), batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=pad_collate, num_workers=2, pin_memory=True)

total_training_steps = (len(train_loader) // ACCUM_STEPS) * NUM_EPOCHS

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, betas=(0.9, 0.98), eps=1e-9, weight_decay=WEIGHT_DECAY)

def cosine_warmup_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    progress = float(current_step - WARMUP_STEPS) / float(max(1, total_training_steps - WARMUP_STEPS))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_ratio = MIN_LR / BASE_LR
    return min_ratio + (1.0 - min_ratio) * cosine_decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_warmup_lambda)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler('cuda')

print(f"Configured Cosine Warmup Schedule: {total_training_steps:,} total steps ({WARMUP_STEPS} warmup steps).")

### Step 8: Accelerated Training with Gradient Accumulation
Runs 25 epochs using **FP16 AMP**, gradient accumulation ($2\times$), and tracking the top 3 checkpoints for Stochastic Weight Averaging (SWA).

In [ ]:
history = []
top_checkpoints = []  # Stores (val_loss, state_dict)
global_start = time.time()

print("Starting Enhanced GPU Training (~10–12 minutes on T4)...\n")
for epoch in range(1, NUM_EPOCHS + 1):
    ep_start = time.time()
    model.train()
    train_loss, train_tokens = 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, (src_ids, tgt_ids) in enumerate(train_loader):
        src_ids, tgt_ids = src_ids.to(DEVICE), tgt_ids.to(DEVICE)
        dec_in = tgt_ids[:, :-1]
        dec_tgt = tgt_ids[:, 1:]

        with torch.amp.autocast('cuda'):
            logits = model(src_ids, dec_in)
            loss = loss_fn(logits.reshape(-1, OR_VOCAB_SIZE), dec_tgt.reshape(-1))
            scaled_loss = loss / ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        n_tok = (dec_tgt != PAD_ID).sum().item()
        train_loss += loss.item() * n_tok
        train_tokens += n_tok

    avg_train_loss = train_loss / train_tokens

    # Validation
    model.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for src_ids, tgt_ids in val_loader:
            src_ids, tgt_ids = src_ids.to(DEVICE), tgt_ids.to(DEVICE)
            dec_in = tgt_ids[:, :-1]
            dec_tgt = tgt_ids[:, 1:]
            with torch.amp.autocast('cuda'):
                logits = model(src_ids, dec_in)
                loss = loss_fn(logits.reshape(-1, OR_VOCAB_SIZE), dec_tgt.reshape(-1))
            n_tok = (dec_tgt != PAD_ID).sum().item()
            val_loss += loss.item() * n_tok
            val_tokens += n_tok

    avg_val_loss = val_loss / val_tokens
    ep_duration = time.time() - ep_start
    curr_lr = scheduler.get_last_lr()[0]

    history.append({"epoch": epoch, "train_loss": avg_train_loss, "val_loss": avg_val_loss, "lr": curr_lr})
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {curr_lr:.2e} | Time: {ep_duration:.1f}s")

    # Save for SWA (top 3 checkpoints)
    cpu_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    top_checkpoints.append((avg_val_loss, cpu_state))
    top_checkpoints.sort(key=lambda x: x[0])
    if len(top_checkpoints) > 3:
        top_checkpoints.pop()

total_elapsed = time.time() - global_start
print(f"\nTraining complete in {total_elapsed/60:.1f} minutes!")

### Step 9: Checkpoint Averaging (SWA) & Beam Search Evaluation
Averages the weights of the top 3 checkpoints, evaluates test set using **Beam Search ($k=4$)**, and saves outputs to `/kaggle/working/`.

In [ ]:
import sacrebleu
from IPython.display import display

# 1. Compute SWA average of top 3 checkpoints
print("Applying Checkpoint Averaging (SWA) over top 3 checkpoints...")
swa_state = {}
for key in top_checkpoints[0][1].keys():
    tensors = [ckpt[1][key].float() for ckpt in top_checkpoints]
    avg_tensor = sum(tensors) / len(tensors)
    orig_dtype = top_checkpoints[0][1][key].dtype
    swa_state[key] = avg_tensor.to(orig_dtype)

model.load_state_dict({k: v.to(DEVICE) for k, v in swa_state.items()})
model.eval()

# Save final checkpoint to working directory
ckpt_path = KAGGLE_WORKING / "scaled_model_best.pt"
torch.save({
    "model_state": swa_state,
    "config": {
        "d_model": D_MODEL,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "n_encoder_layers": N_ENCODER_LAYERS,
        "n_decoder_layers": N_DECODER_LAYERS,
        "dropout": DROPOUT,
        "src_vocab_size": EN_VOCAB_SIZE,
        "tgt_vocab_size": OR_VOCAB_SIZE,
        "max_len": MAX_LEN,
        "tie_weights": TIE_WEIGHTS,
        "architecture": "Pre-LN Scaled Transformer with Weight Tying",
    },
    "best_val_loss": top_checkpoints[0][0],
}, str(ckpt_path))
print(f"Saved checkpoint: {ckpt_path}!")

# Save tokenizers to working directory so they can be loaded anywhere
tok_dir = KAGGLE_WORKING / "tokenizers"
tok_dir.mkdir(parents=True, exist_ok=True)
en_tok.save(str(tok_dir / "en_bpe.json"))
or_tok.save(str(tok_dir / "or_bpe.json"))
print(f"Saved tokenizers to: {tok_dir}!")

# 2. Fast Greedy Decoder (for rapid test set evaluation, ~20s for 2000 pairs)
def greedy_decode(model, src_ids, max_len=MAX_LEN):
    tgt_ids = torch.tensor([[SOS_ID]], dtype=torch.long, device=DEVICE)
    for _ in range(max_len - 1):
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda"):
                logits = model(src_ids, tgt_ids)
        next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tgt_ids = torch.cat([tgt_ids, next_tok], dim=-1)
        if next_tok.item() == EOS_ID:
            break
    return tgt_ids[0].tolist()

# 3. Beam Search Decoder (Width=4, Length Penalty=0.6, 3-gram repetition blocking)
def beam_search(model, src_ids, beam_width=4, length_penalty=0.6, max_len=MAX_LEN):
    beams = [([SOS_ID], 0.0)]
    completed = []
    for _ in range(max_len - 1):
        candidates = []
        all_done = True
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                completed.append((seq, score))
                continue
            all_done = False
            tgt_t = torch.tensor([seq], dtype=torch.long, device=DEVICE)
            with torch.no_grad():
                with torch.amp.autocast(device_type="cuda"):
                    logits = model(src_ids, tgt_t)
            log_probs = F.log_softmax(logits[0, -1, :], dim=-1)
            # 3-gram repetition suppression
            if len(seq) >= 2:
                for p in range(len(seq) - 2):
                    if seq[p:p+2] == seq[-2:]:
                        log_probs[seq[p+2]] = float("-inf")
            topk_p, topk_ids = torch.topk(log_probs, beam_width)
            for k in range(beam_width):
                candidates.append((seq + [topk_ids[k].item()], score + topk_p[k].item()))
        if all_done or not candidates:
            break
        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_width]
    if not completed:
        completed = beams
    completed.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    return completed[0][0]

# 4. Evaluate Test Set
print(f"Decoding full test set ({len(test_data):,} pairs) with Greedy Decoder...")
hyps, refs = [], []
for item in test_data:
    s_tensor = torch.tensor([item["src_ids"]], dtype=torch.long, device=DEVICE)
    out_ids = greedy_decode(model, s_tensor)
    h_text = or_tok.decode(out_ids, skip_special_tokens=True).strip()
    r_text = item["tgt"].strip()
    hyps.append(h_text)
    refs.append(r_text)

bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("\n=======================================================")
print(f"=== Enhanced Scaled Model Corpus BLEU: {bleu.score:.2f} ===")
print("=======================================================")
print(f"BLEU Signature: {bleu.format()}")

# 5. Qualitative Sample Translations (Comparing Greedy vs Beam Search)
print("\n--- Sample Translations (Greedy vs Beam Search) ---")
sample_rows = []
for item in test_data[:5]:
    s_tensor = torch.tensor([item["src_ids"]], dtype=torch.long, device=DEVICE)
    g_ids = greedy_decode(model, s_tensor)
    b_ids = beam_search(model, s_tensor, beam_width=4)
    g_text = or_tok.decode(g_ids, skip_special_tokens=True).strip()
    b_text = or_tok.decode(b_ids, skip_special_tokens=True).strip()
    sample_rows.append({
        "Source": item["src"],
        "Reference": item["tgt"].strip(),
        "Greedy Output": g_text,
        "Beam Search (k=4)": b_text
    })

df_samples = pd.DataFrame(sample_rows)
display(df_samples)

# 6. Save results to JSON
eval_results = {
    "bleu_score": bleu.score,
    "bleu_signature": str(bleu),
    "num_test_examples": len(test_data),
    "samples": sample_rows,
    "techniques": [
        "Pre-LayerNorm Residuals",
        "Embedding & Output Weight Tying",
        "Cosine Annealing Schedule with Warmup",
        "Gradient Accumulation (Effective Batch 256)",
        "Stochastic Checkpoint Averaging (Top 3)",
        "Beam Search (Width 4, Alpha 0.6)"
    ]
}
with open(KAGGLE_WORKING / "scaled_eval_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

with open(KAGGLE_WORKING / "scaled_training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print(f"\nDone! Artifacts saved to {KAGGLE_WORKING}:")
print(f" - {ckpt_path}")
print(f" - {tok_dir}")
print(f" - {KAGGLE_WORKING / 'scaled_eval_results.json'}")
print(f" - {KAGGLE_WORKING / 'scaled_training_history.json'}")
